# Работа по теме "Численное прогнозирование"

## Описание

Работа предполагает самостоятельное исследование данных и построение регрессионных моделей.

Основная задача работы - научиться работать с методами численного прогнозирования, настраивать и отбирать лучшие модели.

Предлагается один набор данных.

Примерные критерии оценки:
- представленные данные изучены и описаны;
- данные предобработаны при необходимости, разделены на выборки;
- верно проведена обработка категориальных данных;
- рассмотрено не менее трех различных методов регрессии;
- создаваемые модели настроены для получения наилучших результатов;
- рассмотрены различные метрики для оценки, при описании результатов метрики верно интерпретированы.

В результате необходимо получить наилучшую модель численного прогнозирования, при этом основную метрику разрешается выбрать самостоятельно, объяснив свой выбор. Также должен быть показан весь процесс выбора и настройки моделей.

## Импорт библиотек

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# Анализ доходов домохозяйств

Набор данных `household_income.csv`.

Описание данных:
- Age: Age of the primary household member (18 to 70 years).
- Education Level: Highest education level attained (High School, Bachelor's, Master's, Doctorate).
- Occupation: Type of occupation (Healthcare, Education, Technology, Finance, Others).
- Number of Dependents: Number of dependents in the household (0 to 5).
- Location: Residential location (Urban, Suburban, Rural).
- Work Experience: Years of work experience (0 to 50 years).
- Marital Status: Marital status of the primary household member (Single, Married, Divorced).
- Employment Status: Employment status of the primary household member (Full-time, Part-time, Self-employed).
- Household Size: Total number of individuals living in the household (1 to 7).
- Homeownership Status: Homeownership status (Own, Rent).
- Type of Housing: Type of housing (Apartment, Single-family home, Townhouse).
- Gender: Gender of the primary household member (Male, Female).
- Primary Mode of Transportation: Primary mode of transportation used by the household member
(Car, Public transit, Biking, Walking).
- Income: Actual annual household income, derived from a combination of features
with added noise. Unit USD

Целевая переменная - Income.

Требуется построить наилучшую модель для прогнозирования доходов домохозяйств.

In [2]:
def metrics(y_true, y_pred, name=''):
    if name:
        print(f'──── {name} ────')
    print(f'R2:   {r2_score(y_true, y_pred):.4f}')
    print(f'MAE:  {mean_absolute_error(y_true, y_pred):>12,.0f}')
    print(f'RMSE: {mean_squared_error(y_true, y_pred)**0.5:>12,.0f}$')
    print(f'MAPE: {mean_absolute_percentage_error(y_true, y_pred):.2%}')
    print()

In [3]:
df = pd.read_csv('household_income.csv')
df.head()

,Age,Education_Level,Occupation,Number_of_Dependents,Location,Work_Experience,Marital_Status,Employment_Status,Household_Size,Homeownership_Status,Type_of_Housing,Gender,Primary_Mode_of_Transportation,Income
0,56,Master's,Technology,5,Urban,21,Married,Full-time,7,Own,Apartment,Male,Public transit,72510
1,69,High School,Finance,0,Urban,4,Single,Full-time,7,Own,Apartment,Male,Biking,75462
2,46,Bachelor's,Technology,1,Urban,1,Single,Full-time,7,Own,Single-family home,Female,Car,71748
3,32,High School,Others,2,Urban,32,Married,Full-time,1,Own,Apartment,Female,Car,74520
4,60,Bachelor's,Finance,3,Urban,15,Married,Self-employed,4,Own,Townhouse,Male,Walking,640210


In [4]:
df.describe()

,Age,Number_of_Dependents,Work_Experience,Household_Size,Income
count,10000.000000,10000.000000,10000.000000,10000.000000,1.000000e+04
mean,44.021700,2.527000,24.858800,3.989600,8.168382e+05
std,15.203998,1.713991,14.652622,2.010496,1.821089e+06
min,18.000000,0.000000,0.000000,1.000000,3.104400e+04
25%,31.000000,1.000000,12.000000,2.000000,6.844600e+04
50%,44.000000,3.000000,25.000000,4.000000,7.294300e+04
75%,57.000000,4.000000,37.000000,6.000000,3.506675e+05
max,70.000000,5.000000,50.000000,7.000000,9.992571e+06


In [5]:
cat_cols = df.select_dtypes(include='object').columns.tolist()

for col in cat_cols:
    print(f'{col:35s}: {list(df[col].unique())}')

Education_Level                    : ["Master's", 'High School', "Bachelor's", 'Doctorate']
Occupation                         : ['Technology', 'Finance', 'Others', 'Education', 'Healthcare']
Location                           : ['Urban', 'Rural', 'Suburban']
Marital_Status                     : ['Married', 'Single', 'Divorced']
Employment_Status                  : ['Full-time', 'Self-employed', 'Part-time']
Homeownership_Status               : ['Own', 'Rent']
Type_of_Housing                    : ['Apartment', 'Single-family home', 'Townhouse']
Gender                             : ['Male', 'Female']
Primary_Mode_of_Transportation     : ['Public transit', 'Biking', 'Car', 'Walking']


In [7]:
X = df.drop('Income', axis=1)
y = df['Income']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

cat_features_ordinal = ['Education_Level']
cat_features_nominal = [col for col in cat_cols if col != 'Education_Level']

preprocessor = ColumnTransformer(
    transformers=[
        ('ordinal', OrdinalEncoder(categories=[["High School", "Bachelor's", "Master's", "Doctorate"]]), cat_features_ordinal),
        ('onehot', OneHotEncoder(handle_unknown='ignore'), cat_features_nominal)
    ],
    remainder='passthrough'
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Shape of processed training data:", X_train_processed.shape)
print("Shape of processed testing data:", X_test_processed.shape)

Shape of processed training data: (8000, 30)
Shape of processed testing data: (2000, 30)


## Линейная ргерессия

In [8]:
numerical_features = X.select_dtypes(include=np.number).columns.tolist()

preprocessor_final = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('ordinal', OrdinalEncoder(categories=[["High School", "Bachelor's", "Master's", "Doctorate"]]), cat_features_ordinal),
        ('onehot', OneHotEncoder(handle_unknown='ignore'), cat_features_nominal)
    ]
)

linear_reg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_final),
    ('regressor', LinearRegression())
])

linear_reg_pipeline.fit(X_train, y_train)

y_pred_lr = linear_reg_pipeline.predict(X_test)

metrics(y_test, y_pred_lr, name='Linear Regression')

──── Linear Regression ────
R2:   0.0056
MAE:     1,102,569
RMSE:    1,771,173$
MAPE: 743.70%



## RandomForest

In [9]:
random_forest_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_final),
    ('regressor', RandomForestRegressor(random_state=42))
])

param_grid = {
    'regressor__n_estimators': [100, 200, 300],
    'regressor__max_features': [0.6, 0.8, 1.0],
    'regressor__max_depth': [10, 20, 30, None]
}

grid_search_rf = GridSearchCV(random_forest_pipeline, param_grid, cv=3, scoring='neg_mean_squared_error', n_jobs=-1, verbose=1)

grid_search_rf.fit(X_train, y_train)

best_rf_model = grid_search_rf.best_estimator_

print("Best parameters found:", grid_search_rf.best_params_)

y_pred_rf = best_rf_model.predict(X_test)

metrics(y_test, y_pred_rf, name='Random Forest (tuned)')

Fitting 3 folds for each of 36 candidates, totalling 108 fits
Best parameters found: {'regressor__max_depth': 10, 'regressor__max_features': 0.6, 'regressor__n_estimators': 300}
──── Random Forest (tuned) ────
R2:   0.0433
MAE:     1,084,321
RMSE:    1,737,299$
MAPE: 750.69%



## Gradient Boosting

In [10]:
gradient_boosting_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_final),
    ('regressor', GradientBoostingRegressor(random_state=42))
])

param_grid_gb = {
    'regressor__n_estimators': [100, 200, 300],
    'regressor__learning_rate': [0.01, 0.05, 0.1],
    'regressor__max_depth': [3, 5, 7]
}

grid_search_gb = GridSearchCV(gradient_boosting_pipeline, param_grid_gb, cv=3, scoring='neg_mean_squared_error', n_jobs=-1, verbose=1)

grid_search_gb.fit(X_train, y_train)

best_gb_model = grid_search_gb.best_estimator_

print("Best parameters found:", grid_search_gb.best_params_)

y_pred_gb = best_gb_model.predict(X_test)

metrics(y_test, y_pred_gb, name='Gradient Boosting (tuned)')

Fitting 3 folds for each of 27 candidates, totalling 81 fits
Best parameters found: {'regressor__learning_rate': 0.01, 'regressor__max_depth': 7, 'regressor__n_estimators': 100}
──── Gradient Boosting (tuned) ────
R2:   0.0331
MAE:     1,088,005
RMSE:    1,746,566$
MAPE: 740.86%



In [12]:
print('\nLinear Regression:')
metrics(y_test, y_pred_lr, name='')

print('\nRandom Forest:')
metrics(y_test, y_pred_rf, name='')

print('\nGradient Boosting:')
metrics(y_test, y_pred_gb, name='')


Linear Regression:
R2:   0.0056
MAE:     1,102,569
RMSE:    1,771,173$
MAPE: 743.70%


Random Forest:
R2:   0.0433
MAE:     1,084,321
RMSE:    1,737,299$
MAPE: 750.69%


Gradient Boosting:
R2:   0.0331
MAE:     1,088,005
RMSE:    1,746,566$
MAPE: 740.86%



## Выводы

RandomForest справился лучше всего с R2 = 0.0433.

Однако все модели показали довольно небольшой R2, что говорит о том, что они не могут достаточно точно предсказывать целевую переменную